# Imaging Basics

Digital image processing and analysis are extensive fields, and we will only be able to cover them in the briefest possible way during this workshop. Similarly, experimental design, microscope performance and function, and statistical analysis will only be touched upon.

In this workshop, we will use Python for image analysis. Please ensure you have a conda environment set up with JupyterLab and the necessary libraries.

**If you haven't already, follow these steps:**
```bash
module load anaconda3/2024.6  # on della
conda create -c conda-forge -n my_jupyter jupyterlab matplotlib -y
conda activate my_jupyter
pip install bioio bioio-nd2
```

This last command will also install scikit-image, the primary image processing library we will be using.

## What imaging can tell us about biological condensates

One instrument used frequently in the Brangwynne lab is a fluorescent microscope, which allows capturing the location of labeled compounds at high magnification and resolution. There are many ways to set up a fluorescent microscope: how and where to illuminate, the excitation wavelength, detector location and type, and the specific optical path. In the most basic setup, a photon excites a fluorescent molecule/protein/tag which emits light at a higher wavelength. The microscope will detect the emitted light and record its position. Two molecules will emit twice as much light as one and the recorded position reflects the physical location of the tag, which is likely "glued" onto a biological component you are interested in learning about.

A good first step in image analysis is to just look at your image. What is your treatment changing? Are any cells special? What do they look like? These simple questions guide the workflow that you aim to build. It's also a good time to let your mind wander and relate these small processes to large physical phenomena. Watching how nucleoli flow and merge was one of the first steps in establishing the idea of biological condensates.

But besides pretty images, what can we learn from a fluorescent microscopy image? More than you might expect! By knowing how much protein is where in a cell, you can determine abundance, partition coefficients, colocalization propensity, mobility (through FRAP or tracking), size and shape (related to the surface tension and other interactions), and much more. One of the challenges with designing experimental acquisitions and analyses is picking what metric you want to monitor and test.

By the end of this workshop, you can find a cell and measure the intensity within it. Should you look at the mean, max, minimum, median, variance of the core, the rim, just outside the object...? Even with that short list, you'll have 15 different measurements for each cell. The incorrect thing to do is measure all of those, test them for significance, then decide which metric is important. The first reason is that you are effectively p-value hacking. If I have 100 measurements and test for significance at 95% confidence, 5 of those measures will be significantly different just by chance. So if you must measure everything, be sure to include a multiple testing correction. The second reason is many of those metrics will not be biologically meaningful and only act to increase your processing time and inflate your p-values with multiple testing. Maybe mean intensity is just barely significantly different with treatment. Using the one measurement you have a minor change but after multiple testing correction with 99 other measures it will be non-significant. Alternatively, say you determine the minimum rim intensity is significantly different. Does that provide a biologically interpretable change to advance your hypothesis? In either case, knowing what you want to measure and what it will tell you can prevent both issues.

Another challenge is that without very careful controls and calibrations, most of the measurements will be semiquantitative, meaning most of your results will be comparisons with other images. You can say "The treatment increases the intensity by 10%" but not "The concentration of protein X in the cell changed from 10 pM to 11 pM upon treatment".

Finally, remember that each setting on the fluorescent microscope will change your image. Altering the excitation intensity will change the emission intensity. Someone knocking the stage may cause different channels to be misaligned. Changing the lens for magnification will clearly change the pixel size, but the lens may have different material properties or a numerical aperture which affects the amount of collected light and resolving power of the system. Outside of the microscope, cellular heterogeneity introduces a plethora of variance from cell cycle state, viral load, time since last media change, time left in ambient light/temperature/air, and passage number. In imaging a 96-well plate, the first and last wells may vary greatly just because they were the first and last to be measured! Some of these sources of variance can be controlled, and should be! Use the same profile on the microscope and try to perform experiments exactly the same except for your experimental perturbation. For sources of variance you can't control, try to sample and incorporate that variance into your analysis. Keep careful notes of any variables that could lead to batch effects; you can always test for their contributions to your measurements independent of the experimental perturbation.

For example, you suspect the microscope isn't exactly the same on different days and you think your protein signal is decreasing over time. Be sure to use the same settings on the microscope as much as possible but record all the relevant metadata that could change (e.g. date and time of acquisition). For the change over time, try to randomize where in your well plate the treatment and control are. While a totally random layout is best, it can be difficult experimentally to keep track of random well positions. A reasonable compromise would be to intersperse control and treatment wells across columns and change the order on different days. When performing statistical tests, you can include date and column as covariates to test (and account for) their effects on results. *Overall, control the variance you can and sample the variance you can't control.*

## Why do I need a computer for this?

The human visual system is truly remarkable in its ability to detect patterns, motion and determine what objects are. From an early age, you could follow a moving object, tell faces apart, and decide if food would be yucky or yummy. In the entirety of human history, it was important to find prey and predators, track their movements, remember faces of people you trust, and tell poisonous foods from nutritious. To survive, we became very good at those things but at the cost of losing objective measurements of images. Even before your brain consciously "sees" an image from your eye, the light entering your eye has been processed. The optic nerve is in the center of your retina and creates a blind spot in your field of view. Your brain combines the images from each eye to try and hide it from your consciousness. With one eye closed, your brain still tries to fill in the spot based on the patterns around it and by constantly moving the eye. The processing in the brain is optimized to determine relative changes in intensity (on a log scale), find edges of objects, and group things into patterns. Because this happens subconsciously and sometimes in the nerves of your eyes, even knowing you have these processing artifacts doesn't make them disappear. This leads to numerous visual illusions which highlight how your eyes and brain can trick you into seeing things that aren't present or making connections from noise. Even between people, there can be disagreements on what an image shows because of their prior experiences or expectations.

So while looking at an image can give you intuition and guidance on what an image is showing, it cannot replace objective quantification from a computer. While looking through the following optical illusions, think about how you could quantify what is happening in a way that would eliminate the illusion. Next time you look at an image, try to remember some of these tricks and think if your brain might be tricking you!

Images from "The Image Processing Handbook, 7th Edition" by John C Russ and F. Brent Neal and Wikipedia.

### The perception of color is context and person dependent

You may remember the blue/black or gold/white dress from a few years ago where people couldn't agree what color a dress was due to perceptions of shadows and light in the image. There are several other examples of optical illusions where the background or context of a color changes how bright or even what color we think is in the image.

<img src="images/Checker_shadow.png" alt="Shadows on a checkerboard" width="500"/>

Here the cylinder casts a shadow on a checkerboard and nothing seems off. What may be surprising is the squares marked "A" and "B" are the same color, even though A looks much darker!

<img src="images/Gradient.png" alt="Gradient background" width="500"/>

While the context of the shadow changed the perceived color on the last image, here the presence of a gradient background makes the left side of the center bar look lighter than the right, though again they are the same shade of grey. Another phenomenon worth looking into is Mach bands, where the boundary of two colors is perceived to have a sharper difference in color than is present.

<img src="images/Blue_eye.png" alt="Color context" width="500"/>

In addition to shades of grey, the perception of color depends on the context around the object. Here the red background on the left side of the character's face makes the left eye appear blue, though it is really the same color as the right eye. I had to crop it out to check!

<img src="images/Leaves.png" alt="Color blindness" width="500"/>

Of course our perception is not the only difference between observers. Around 10% of males have some form of color blindness.

a. Original color image

b. Simulated deuteranopia (red/green)

c. Simulated tritanopia (blue/yellow)

d. Simulated achromatopsia (total color blindness)

### Estimating areas dependes on context, expectations and shape

Equally important in biological images is the prediction of area of objects. Again your brain can play tricks on you by taking a shape's context into account when trying to determine the relative size of objects.

<img src="images/Mond-vergleich.svg" alt="Mond vergleich" width="500"/>

The Mond vergleich illusion shows two orange circles that are the same size, though the one on the right is perceived as larger because of the smaller surrounding grey circles.

<img src="images/Shapes.png" alt="Different shapes" width="500"/>

Each object in the image has the same area, but because of the different shapes and boundaries the right two appear larger and the oval smaller.

<img src="images/Tables.png" alt="Shepart tables" width="500"/>

Both table tops are made of the same parallelogram, just rotated by 90%. The addition of the table legs gives the illusion of depth and causes your mind to see the left as thinner than the right table is deep.

### Patterns can confuse the brain

This is probably less important for condensate biology, but maybe in studying cytoskeletal arrangements you could generate similar patterns. Mostly it's fun to see your brain play tricks!

<img src="images/Zollner.png" alt="Zollner lines" width="500"/>

The cross hatching of the diagonal lines make you perceive the lines as bending into each other, when actually they are parallel.

<img src="images/Drift.png" alt="Drift illusion" width="500"/>

The alternating lines appear to be moving right or left of each other.

### "Seeing is inference from incomplete information."

The above images were designed to highlight different biases and neural processes involved with vision. In the lab, you probably won't generate images that double as optical illusions, but it is important to remember that just because something looks a certain way doesn't mean it is that way. You have a subjective view of reality in general and when viewing data from an experiment you have spent months preparing or a final experiment before a conference or publication the added emotional pressures can further distort what the image shows.

So while seeing is believing, having an objective algorithm for measuring features in an image is the only way to provide a statistical backing to qualitative observations. When the results of the analysis don't match your expectations, it could be the analysis is flawed, your perception is biased, or that the few "interesting" cells you've been poring over are not as representative as you hoped. Our visual system is well tuned to help us navigate and survive our world, not for detecting a 5% change in intensity in microscopy images. This is important to remember, because you are exceptionally good at inferring things from images without even knowing it.

<img src="images/Writing.png" alt="Word context" width="500"/>

While the last word in each line is identical, you probably have little trouble in reading the sentence since your brain uses the surrounding words to resolve the ambiguity.

## Images in python

Python is among the most popular languages in the physical sciences because it is easy to learn and has a large ecosystem of libraries for scientific computing. With the right library and a few lines of code you can get your analysis done quickly and correctly. We won't cover most of the basics as there are copious options for learning more about Python and each dependency we will use. The goal isn't to get you proficient with writing image analysis code in Python but to help you build intuition on what some higher level functions are doing, like enhancement and blob finding.

For your day to day analysis, I highly recommend using a pre-built package or tool when you can. It will save you time in replicating existing algorithms and ensure you aren't making any logical or programming mistakes. 90% of your analysis will already exist somewhere, perhaps as Fiji plugins or CellProfiler modules. The last 10% can be added with custom code so you don't have to write code for filtering and blob finding over and over again.

### Opening and examining images

Let's start with a simple color image. Images from your cell phone are stored as a matrix of unsigned integers where each byte corresponds to a channel: red, green, or blue. The display on your computer works in an additive color space, where each RGB color activates a specific receptor in your eye. By mixing the relative amounts of each monochromatic light, you interpret different colors. Red and green make yellow, blue and green make cyan, and red and blue make magenta. By emitting equal amounts of each channel, you get shades of grey and ultimately white. In contrast, when mixing paints in elementary school, you were working in a subtractive color space where increasing amounts of pigments absorbed more of a certain spectrum. There, mixing the primary colors of red, yellow, and blue gave orange, green, purple, and all together brown (and eventually black). In multichannel fluorescence images, a standard convention is to map a channel onto each red, green, and blue to show a pseudo-color image. There, it's important to remember what mixtures of channels generate which colors. Also note that the assignment of a channel to a color on the screen is relatively arbitrary, though following previous work is a good idea.

Let's open a color image and see what it's made of:

In [ ]:
# import necessary libraries
import skimage
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import bioio

In [ ]:
# get a sample image from skimage
image = skimage.data.astronaut()
plt.imshow(image)

In [ ]:
# for presentation, sometimes it's nice to also adjust the size and remove the axis labels
fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(image)
ax.set_axis_off()

In [ ]:
# look at the shape and contents of the top left corner
print(f'Image is of type {type(image)}')
print(f'Image has the shape {image.shape}')
print(f'Top 4x4 area, red channel')
print(image[:4, :4, 0])

# show the top 100x100 pixel area and add a title
fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(image[:100, :100, :])
ax.set_axis_off()
ax.set_title('Crop of flag')

When cropping, there is an issue where the coordinate displayed on the image is transposed for how you think, e.g., the y-axis is used for the first slice of the array. This is due to differences in how images are displayed on a coordinate system vs. stored as a matrix. Let's try to put a dot on the astronaut's nose and then crop out the face to see what I mean.

From the first image (with axes labels), the face is between x=150:300 and y=0:200, and the nose is about at (225, 125).

In [ ]:
# let's try to use our first instinct for x and y
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].set_title('Dot on the nose')
axes[0].imshow(image)
axes[0].plot(125, 225, 'ro')
axes[1].set_title('Crop of face')
axes[1].imshow(image[150:300, 0:200])

In [ ]:
# Terrible job, let's transpose x and y
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].set_title('Dot on the nose')
axes[0].imshow(image)
axes[0].plot(225, 125, 'ro')
axes[1].set_title('Crop of face')
axes[1].imshow(image[0:200, 150:300])

Also note how the coordinate system of the cropped image is also changed; the top left is always 0, 0.

The reason for the apparent difference is that the matrix is indexed by row, column order. The 10th row of the matrix corresponds to y=10 in the image, while the columns move along the x direction.

### Splitting and changing colors

Now let's mess with the colors. First we can deconstruct the image into its separate channels then reassign them to a different position in the array.

In [ ]:
# create a multipanel figure.  Note that the number of rows and columns is reversed from the figsize argument
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# iterate along the different subplot axes, the image (along the last, color channel), and titles for each subplot
for ax, channel, channel_name in zip(axes, np.rollaxis(image, -1), ('red', 'green', 'blue')):
    ax.imshow(channel)
    ax.set_axis_off()
    ax.set_title(channel_name)

By default, Matplotlib uses a viridis colormap, but you may prefer grey or matching the specific color. You can create an RGB image by setting the appropriate color channel in an empty matrix. Here are some examples:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Grey Scale')
for ax, channel, channel_name in zip(axes, np.rollaxis(image, -1), ('red', 'green', 'blue')):
    ax.imshow(channel, cmap='grey')
    ax.set_axis_off()
    ax.set_title(channel_name)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))    
fig.suptitle('RGB')
# need to also enumerate over the zipped output to keep track of which color index to set on the temporary single-color image
for channel_index, (ax, channel, channel_name) in enumerate(zip(axes, np.rollaxis(image, -1), ('red', 'green', 'blue'))):
    # create a black (0) array that's the same type and size as the input image
    img = np.zeros_like(image)
    # set just the desired channel in the temporary img
    img[..., channel_index] = channel
    ax.imshow(img)
    ax.set_axis_off()
    ax.set_title(channel_name)

In [ ]:
# now we can swap some channels.  First make a copy of the image to keep the original
swapped_image = image.copy()
# next swap the red[0] and green[1] channels
swapped_image[..., 0], swapped_image[..., 1] = swapped_image[..., 1], swapped_image[..., 0]

fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(swapped_image)
ax.set_axis_off()
ax.set_title('$Red \leftrightarrow Green$')

## Biological images with bioio

While we could keep looking at Eileen Collins for more processing, let's open a sample nd2 image from the lab to see how to deal with image formats and more than 3 channels.

First, bioio has different modules or plugins for most biological image formats, including TIFFs, CZI, and ND2 images. You may have to install extras to use them, like the ND2 installation we did above. The interface for opening and looking at images is slightly different from above, but the underlying data is still a matrix of numbers.

In [ ]:
cell_image = bioio.BioImage('images/sample.nd2')
# the cell image object doesn't yet have access to the raw data, but it can give you lots of metadata
print(f'channel names: {cell_image.channel_names}')
print(f'dims: {cell_image.dims}')
print(f'shape: {cell_image.shape}')
print(f'scenes: {cell_image.scenes}')

First, the file metadata includes information that was input to the microscope during acquisition. See the `cell_image.metadata` for a full accounting of what's saved. Common properties are pulled out into other properties and used during object creation. One example is the channel names, which can help determine what channel corresponds to what index.

Notice how the `dims` property has a name associated with each number, while the `shape` property does not. It's highly recommended to use the `dims` to access regions, to avoid confusion over x/y and if you acquire a 4-frame, 4-channel, 4-z-stack image. We will go over how to do that next.

Finally, if your image contains multiple scenes, you can select them prior to loading the image. By default, the first scene will be retrieved.

To start, we will use the `xarray_data` property, which has several useful features. An xarray is like a NumPy array that also has named dimensions, so you can access them by name. If you prefer raw NumPy, you can use the similar `data` property, which only returns the selected scene as a NumPy array. An alternative to data is the `get_image_data` function, which also accepts arguments for the order of channels to return and which regions to obtain. You could use the data property and slice afterwards, but with `get_image_data`, you save on memory by only loading the parts of the image you need.

Using that, let's look at the image for channels 1:3 in RGB.

In [ ]:
# to start, let's use a numpy array
plt.imshow(
    cell_image.get_image_data(
            'YXC',  # to display rgb images, the channel dimension has to be last.
            T=0, Z=0,  # fix these as the first index.  Can also omit and squeeze the result
            C=range(1,4),  # get channels 1, 2, and 3
            Y=slice(0, -1, 10),  # only read every 10th pixel for showing an overview of the image
            X=slice(0, -1, 10),
        )
)

Which looks terrible since matplotlib expects rgb data to be between 0 and 1 while ours goes from 95 to 5009.  Let's switch to an xarray and do the normalization.

In [ ]:
# read entire scene to memory.  squeeze removes singleton dimensions (T and Z here)
xcell_image = cell_image.xarray_data.squeeze()
# now we want to determine the min and max values to normalize for display
# passing in X and Y tells xarray we want the min/max over those dimensions.  The result is a min/max for each channel
img_min, img_max = xcell_image.min(['X', 'Y']), xcell_image.max(['X', 'Y'])
# Normalize the image between 0 and 1, xarray takes care of keeping the sizes and columns straight
xcell_normed = (xcell_image - img_min) / (img_max - img_min)
# select every 10th pixel and the last 3 channels
to_show = xcell_normed.isel(X=slice(0, -1, 10), Y=slice(0, -1, 10), C=range(1, 4))
plt.imshow(
    to_show.transpose(..., 'C')  # need to put the color channel at the end
)

Because we have access to the image metadata, we can also select channels by name and pixels by their size in um instead of pixel indices.  To do so, use the function `sel` instead of `isel`.  Lastly, let's try to crop out the cell that's around x=60, y=100.

In [ ]:
# we need to multiply the bounds by 10 since the image above was decimated 10x.  Note that the order doesn't matter since X and Y are named explicitly
to_show = xcell_normed.isel(X=slice(400, 1000, 10), Y=slice(500, 1200, 10), C=range(1, 4))
plt.imshow(
    to_show.transpose(..., 'C')  # need to put the color channel at the end
)

In [ ]:
# since we have a small image we can look at it in full resolution, show EU in another subplot
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
xy_slice = dict(X=slice(400, 1000), Y=slice(500, 1200))
to_show = xcell_normed.isel(**xy_slice, C=range(1, 4))
axes[0].imshow(
    to_show.transpose(..., 'C')  # need to put the color channel at the end
)
axes[0].set_title('DFC  FC  GC  Image')
axes[1].imshow(xcell_normed.isel(**xy_slice, C=0), cmap='gray')
axes[1].set_title('EU  Image')

## Making measurements

Now that we can see an image, we should try to measure something about it.  We will try to measure:
- Size and shape of nucleoli
- Enrichment of EU in the nucleolus vs nucleus
- Number of FCs per nucleolus

Most analysis procedures require a combination of intensity images (what we've been looking at) and masks or label images.  Masks are binary images where each pixel is True if it is part of the foreground and False otherwise.  Label images are typically generated from masks, but encode unique objects with an integer.  0 is reserved for the background and each object has a unique integer.  As you work, you will generate and combine masks and images to produce the measurements you care about.  Looking at a nucleolus, think about what makes it different from the background.  Something that should jump out immediately is the GC channel is much brighter.  Let's try to quantify that with a histogram.

In [ ]:
cell_image = bioio.BioImage('images/sample.nd2')
# read entire scene to memory.  squeeze removes singleton dimensions (T and Z here)
xcell_image = cell_image.xarray_data.squeeze()

# now we want to determine the min and max values to normalize for display
# passing in X and Y tells xarray we want the min/max over those dimensions.  The result is a min/max for each channel
img_min, img_max = xcell_image.min(['X', 'Y']), xcell_image.max(['X', 'Y'])
# Normalize the image between 0 and 1, xarray takes care of keeping the sizes and columns straight
xcell_normed = (xcell_image - img_min) / (img_max - img_min)

# check the intensity histogram of the 4th channel (index 3)
plt.hist(xcell_image.isel(C=3).stack(stacked=[...]), bins=100)
plt.ylim(0, 1e4)

We have truncated the first background peak with intensities < 200 by rescaling the plot.  Still it's not clear if the threshold should be 300, 500, or maybe 800.  We can try them all:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
gc_img = xcell_image.isel(C=3)
for ax, threshold in zip(axes, (300, 500, 800)):
    ax.imshow(gc_img > threshold)  # show a binary image
    ax.set_title(f'threshold = {threshold}')

So which is better?  It depends on your goal.  If you wanted every thing that could possibly be a GC, maybe 300 or less.  If you are more stringent you may want to use 800 to just get the 3 most intense GC objects.  And of course, manually selecting a threshold can be biased, slow, and inaccurate.  It's much better to use an algorithm to pick a threshold.  Thankfully skimage has several implemented for you.  Here are a few options with the thresholds shown on the plot so you can see what is picked.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
intensities = gc_img.stack(stacked=[...])
for ax in axes:
    ax.hist(intensities, bins=100)
    ax.set_ylim(0, 1e4)

axes[0].set_title('Manual')
axes[0].axvline(300, c='r')
axes[0].axvline(500, c='r')
axes[0].axvline(800, c='r')

threshold = skimage.filters.threshold_otsu(gc_img.to_numpy())
axes[1].set_title(f'Otsu: threshold = {threshold:0.0f}')
axes[1].axvline(threshold, c='r')

threshold = skimage.filters.threshold_mean(gc_img.to_numpy())
axes[2].set_title(f'Mean: threshold = {threshold:0.0f}')
axes[2].axvline(threshold, c='r')

It looks like the Otsu threshold is matching the leanient threshold we manually checked above.  The mean threshold would take far too much of the image.  You can also try every threshold method!

In [ ]:
skimage.filters.try_all_threshold(gc_img.to_numpy(), figsize=(10, 8), verbose=False)

Pick your favorite and read up on the algorithm so you know how it works and why it works in a specific case (or not).  Let's stick with Otsu and go over labeling and filtering our putative GC objects.  Due to noise in the intensity image, you will likely end up with small features (sometimes single pixels) or holes in your objects; the 500 intensity threshold above shows a few such examples.  It's also a good idea to remove any objects that are touching the boundaries of your image since they are not completely in frame.  Once your binary image is cleaned up, you can label adjacent pixels that are in the foreground as belonging to the same object.  Note that objects which are touching **at all** will be grouped together.  More advanced segmentation methods exist, but we won't develop them here.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 18))
axes = axes.flatten()

axes[0].set_title('Original Image')
axes[0].imshow(gc_img, cmap='grey')

mask = gc_img.to_numpy() > skimage.filters.threshold_otsu(gc_img.to_numpy())
axes[1].set_title('Otsu Threshold')
axes[1].imshow(skimage.color.label2rgb(mask))

mask = skimage.morphology.remove_small_objects(mask, min_size=50)
axes[2].set_title('Remove small objects')
axes[2].imshow(skimage.color.label2rgb(mask))

mask = skimage.morphology.remove_small_holes(mask, area_threshold=50)
axes[3].set_title('Remove small holes')
axes[3].imshow(skimage.color.label2rgb(mask))

mask = skimage.segmentation.clear_border(mask)
axes[4].set_title('Remove boundary objects')
axes[4].imshow(skimage.color.label2rgb(mask))

mask = skimage.measure.label(mask)
axes[5].set_title('Label objects')
axes[5].imshow(skimage.color.label2rgb(mask))

nucleoli_labels = mask.copy()

With our objects in hand, we can start measuring some properties of what we found.  Let's see all the things we can measure and look at how to use that information to further filter our objects before moving on.  See [the documentation](https://scikit-image.org/docs/0.25.x/api/skimage.measure.html#skimage.measure.regionprops) for a list of all available properties

In [ ]:
# correct background of the gc image with a rolling ball estimate
bkg_gc_img = gc_img.to_numpy() - skimage.restoration.rolling_ball(gc_img.to_numpy())

nucleoli_measurements = skimage.measure.regionprops_table(
    label_image=mask,
    intensity_image=bkg_gc_img,
    properties=[
        'label',  # which object this is
        'area',  # size in pixels
        'centroid',  # xy position of this object
        'eccentricity',  # how eccentric the object is
        'intensity_mean',  # average intensity
        'intensity_std',  # stdev of intensity
        'perimeter',
    ],
)

# we can also add a derived measurement, like the roundness
nucleoli_measurements['roundness'] = 4 * np.pi * nucleoli_measurements['area'] / nucleoli_measurements['perimeter']**2
nucleoli_measurements['intensity_total'] = nucleoli_measurements['intensity_mean'] * nucleoli_measurements['area']
nucleoli_measurements = pd.DataFrame(nucleoli_measurements).rename(columns={'centroid-0': 'centroid-x', 'centroid-1': 'centroid-y'})
nucleoli_measurements.head()

In [ ]:
# let us look at histograms of the roundess and area to pick a few thresholds
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].hist(nucleoli_measurements['roundness'])
axes[0].set_title('Roundness')
axes[1].hist(nucleoli_measurements['area'], bins=20)
axes[1].set_title('Area')

In [ ]:
# label cells that are small and not round
# make a new mask to hold the result
odd_objs = np.zeros_like(mask)  # will be dtype int
odd_objs[
    # look at the pixel locations where the labeled cells (mask) have roundness < 0.7
    np.isin(mask, nucleoli_measurements.loc[nucleoli_measurements['roundness']< 0.7, 'label'])
] += 1  # add one to label them
odd_objs[
    np.isin(mask, nucleoli_measurements.loc[nucleoli_measurements['area'] < 1000, 'label'])
] += 2  # add two to label them.  Cells that are small and not round will have a value of 3

# show original label as well
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(skimage.color.label2rgb(mask))
axes[1].imshow(skimage.color.label2rgb(odd_objs))

In [ ]:
# remove the object that is both small and not circular (top middle)
mask[odd_objs == 3] = 0  # 0 is background

# and redo our measurements
nucleoli_measurements = skimage.measure.regionprops_table(
    label_image=mask,
    intensity_image=bkg_gc_img,
    properties=[
        'label',  # which object this is
        'area',  # size in pixels
        'centroid',  # xy position of this object
        'eccentricity',  # how eccentric the object is
        'intensity_mean',  # average intensity
        'intensity_std',  # stdev of intensity
        'perimeter',
    ],
)

# we can also add a derived measurement, like the roundess
nucleoli_measurements['roundness'] = 4 * np.pi * nucleoli_measurements['area'] / nucleoli_measurements['perimeter']**2
nucleoli_measurements['intensity_total'] = nucleoli_measurements['intensity_mean'] * nucleoli_measurements['area']
nucleoli_measurements = pd.DataFrame(nucleoli_measurements).rename(columns={'centroid-0': 'centroid-x', 'centroid-1': 'centroid-y'})
nucleoli_measurements.head()

Next, we will estimate the nuclear area of each GC by looking at the rim around it. This is accomplished by first dilating each object and then removing the center nucleolar area.  Because the boundary is ill defined, we will also remove some of the nucleolar boundary before making our measurements.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 18))
axes = axes.flatten()

axes[0].imshow(skimage.color.label2rgb(mask))
axes[0].set_title('Original Labels')

# dilate the mask by 43 pixels
dilated_mask = skimage.morphology.dilation(mask, footprint=skimage.morphology.disk(43))
axes[1].imshow(skimage.color.label2rgb(dilated_mask))
axes[1].set_title('Dilated Labels')

# remove the original mask with an added dilation of 3 pixels to get rid of some boundary
rim = dilated_mask.copy()
rim[
    # we dilate 3 pixels then look for pixels which are not 0 (not background)
    skimage.morphology.dilation(mask, footprint=skimage.morphology.disk(3)) != 0
] = 0  # those are set to 0 in the dilated mask to produce the rim
axes[2].imshow(skimage.color.label2rgb(rim))
axes[2].set_title('Rim')

# erode the original mask by 3 pixels to estimate the core area
core = skimage.morphology.erosion(mask, footprint=skimage.morphology.disk(3))
axes[3].imshow(skimage.color.label2rgb(core))
axes[3].set_title('Core')

# show an overlay, because it's fun
overlay = core + rim
# find foreground pixels (rim != 0) and offset by the maximum value of core
overlay[rim != 0] += core.max()
axes[4].imshow(skimage.color.label2rgb(overlay))
axes[4].set_title('Rim + Core')

A few important notes about the above analysis:
- The choice of parameters for the boundary region and dilation are up to you.  It's reasonable to use a symmetric erosion/dilation of the original mask.  Make sure you are not losing your smallest objects by eroding them completely
- The size of the boundary will vary with the original object size.  Larger objects will have larger boundaries.
- Overlapping boundaries are resolved by the index of the original object, somewhat arbitrarily.  You could try to exclude or double count them
- While the image above doesn't show it, the core and rim have matching labels

Now we can measure those additional masks and join them to our existing data table

In [ ]:
# we only care about the intensity measurements
rim_measurements = skimage.measure.regionprops_table(
    label_image=rim,
    intensity_image=bkg_gc_img,
    properties=[
        'area',  # size in pixels
        'intensity_mean',  # average intensity
        'intensity_std',  # stdev of intensity
        'label',  # needed for joining
    ],
)
rim_measurements['intensity_total'] = rim_measurements['intensity_mean'] * rim_measurements['area']
rim_measurements = pd.DataFrame(rim_measurements)

# want to merge the resulting table on the label, which should match between images
# we also give the rim measurements a new suffix while the original measurement is left alone
merged_measurements = nucleoli_measurements.merge(rim_measurements, on='label', suffixes=('', '_rim'))

# repeat for the core
core_measurements = skimage.measure.regionprops_table(
    label_image=core,
    intensity_image=bkg_gc_img,
    properties=[
        'area',  # size in pixels
        'intensity_mean',  # average intensity
        'intensity_std',  # stdev of intensity
        'label',  # needed for joining
    ],
)
core_measurements['intensity_total'] = core_measurements['intensity_mean'] * core_measurements['area']
core_measurements = pd.DataFrame(core_measurements)

# now merging with the measurement that is merged with rim already
merged_measurements = merged_measurements.merge(core_measurements, on='label', suffixes=('', '_core'))

# can estimate an enrichment as mean core intensity / rim
merged_measurements['enrichment'] = merged_measurements['intensity_mean_core'] / merged_measurements['intensity_mean_rim']
merged_measurements.head()

Let's see if the enrichment is correlated to size.  No reason to think it is, but we have to plot something.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(merged_measurements['area'], merged_measurements['enrichment'])
ax.set(xlabel='Area (px)', ylabel='Enrichment')

## Object segmentation in more challenging systems

Nucleoli are a good place to start because they are fairly distinct and have clear boundaries.  Now we will move to two more challenging targets, nuclei from the EU image and FCs within the GCs.  Once segmented, you can measure properies in a nearly identical way but you need to be careful about assigning the correct nucleoli to its nucleus and FCs to nucleoli.  We will start with nuclei and move to FC finding.

The problem with finding nuclei in this image is the intensity is not uniform throughout.  If we try several segmentation schemes you can see none do a very good job of finding just the nuclei.

In [ ]:
cell_image = bioio.BioImage('images/sample.nd2')
# read entire scene to memory.  squeeze removes singleton dimensions (T and Z here)
xcell_image = cell_image.xarray_data.squeeze()

# now we want to determine the min and max values to normalize for display
# passing in X and Y tells xarray we want the min/max over those dimensions.  The result is a min/max for each channel
img_min, img_max = xcell_image.min(['X', 'Y']), xcell_image.max(['X', 'Y'])
# Normalize the image between 0 and 1, xarray takes care of keeping the sizes and columns straight
xcell_normed = (xcell_image - img_min) / (img_max - img_min)

nuclei_image = xcell_normed.isel(C=0).to_numpy()
skimage.filters.try_all_threshold(nuclei_image, figsize=(10, 8), verbose=False)

We can try to improve things by applying filters to the underlying image, by blurring and median filtering.

Gaussian blurs are fast and can remove some of the noise and roughness of the original image but will also blur the boundary of the image.  Median filtering takes the median value in a sliding window of the original image.  This is generally slower but preserves boundaries much better.  The concept of filtering, convolutions and their kernels is a large part of digital image processing that we are mostly glossing over.  Some concepts will come back up when we find FCs but notice they have their own theoretical background you should examine further.

In either case, we are applying these filters *solely* to make the job of nuclei segmentation easier.  You should not use the resulting images for measuring intensities (ever) or for publication (unless you clearly specify in the figure caption what you've changed).  Let's try a few filters and see how Otsu segmentaiton performs.  We will crop out just a few cells for display.

In [ ]:
test_img = nuclei_image[500:1500, 500:1500]
fig, axes = plt.subplots(5, 2, figsize=(12, 30))
axes = axes.flatten()

ind = 0
axes[ind].imshow(test_img, cmap='grey')
axes[ind].set_title('Original Image')
axes[ind+1].imshow(skimage.color.label2rgb(test_img > skimage.filters.threshold_otsu(test_img)))
axes[ind+1].set_title('Otsu Threshold')
ind +=2

img = skimage.filters.gaussian(test_img, sigma=5)
axes[ind].imshow(img, cmap='grey')
axes[ind].set_title('Gaussian sigma=5')
axes[ind+1].imshow(skimage.color.label2rgb(img > skimage.filters.threshold_otsu(img)))
axes[ind+1].set_title('Otsu Threshold')
ind +=2

img = skimage.filters.gaussian(test_img, sigma=20)
axes[ind].imshow(img, cmap='grey')
axes[ind].set_title('Gaussian sigma=20')
axes[ind+1].imshow(skimage.color.label2rgb(img > skimage.filters.threshold_otsu(img)))
axes[ind+1].set_title('Otsu Threshold')
ind +=2

img = skimage.filters.median(test_img, footprint=skimage.morphology.disk(5))
axes[ind].imshow(img, cmap='grey')
axes[ind].set_title('Median radius=5')
axes[ind+1].imshow(skimage.color.label2rgb(img > skimage.filters.threshold_otsu(img)))
axes[ind+1].set_title('Otsu Threshold')
ind +=2

img = skimage.filters.median(test_img, footprint=skimage.morphology.disk(20))
axes[ind].imshow(img, cmap='grey')
axes[ind].set_title('Median radius=20')
axes[ind+1].imshow(skimage.color.label2rgb(img > skimage.filters.threshold_otsu(img)))
axes[ind+1].set_title('Otsu Threshold')
ind +=2

A 20 radius median filter gets close, but it merges nuclei that are close together and the outside is fairly rough.  We can try to resolve those with an erosion followed by an equal dialtion, also called a opening.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes=axes.flatten()
median_filtered = skimage.filters.median(test_img, footprint=skimage.morphology.disk(20))
axes[0].imshow(img, cmap='grey')
axes[0].set_title('Median radius=20')

thresholded = median_filtered > skimage.filters.threshold_otsu(median_filtered)
axes[1].imshow(skimage.color.label2rgb(thresholded))
axes[1].set_title('Otsu Threshold')

opened = skimage.morphology.binary_opening(thresholded, footprint=skimage.morphology.disk(50))
axes[2].imshow(skimage.color.label2rgb(opened))
axes[2].set_title('Open radius 50')

opened = skimage.morphology.binary_opening(thresholded, footprint=skimage.morphology.disk(100))
axes[3].imshow(skimage.color.label2rgb(opened))
axes[3].set_title('Open radius 100')

That looks good enough.  Some other things to try would be a cycle of blur/sharpen to get more defined edges or use the original image intensity to resolve any merged objects.

Now we can look at the entire image.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 18))
axes = axes.flatten()

axes[0].set_title('Original Image')
axes[0].imshow(nuclei_image, cmap='grey')

median_filtered = skimage.filters.median(nuclei_image, footprint=skimage.morphology.disk(20))

mask = median_filtered > skimage.filters.threshold_otsu(median_filtered)
axes[1].set_title('Otsu Threshold')
axes[1].imshow(skimage.color.label2rgb(mask))

mask = skimage.morphology.binary_opening(mask, footprint=skimage.morphology.disk(100))
axes[2].set_title('Closing 100 pixels')
axes[2].imshow(skimage.color.label2rgb(mask))

mask = skimage.morphology.remove_small_holes(mask, area_threshold=50)
axes[3].set_title('Remove small holes')
axes[3].imshow(skimage.color.label2rgb(mask))

mask = skimage.segmentation.clear_border(mask)
axes[4].set_title('Remove boundary objects')
axes[4].imshow(skimage.color.label2rgb(mask))

mask = skimage.measure.label(mask)
axes[5].set_title('Label objects')
axes[5].imshow(skimage.color.label2rgb(mask))

nuclei_labels = mask.copy()

Still not perfect.  The cells we had been looking at are grouped together in the entire image because the Otsu threshold depends on the histogram of intensities from the entire image.  Some options to resolve include:
- Lower the threshold by a tuned factor (e.g. threshold *= 0.9)
- Apply Otsu in an adaptive/windowed fasion
- Use the intensity image to resolve merged objects and recall the boundary
- Use another segmentation method (stardist or cellpose)
- If clear nuclear boundaries are important for the experiment, consider redoing with a dedicated (cleaner) nuclear dye to make the analysis more robust

For now we will assume this is working adequately and look into how we map the nucleoli onto the parent nucleus object.  The first, simple way is to use the nucleli label image as a mask onto the parent nuclear image.  The problem with that is if a nucleoli was found at a boundary it may not have a unique parent.  In the second method, we will find the most common parent pixel for each nucleoli.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

axes[0].imshow(skimage.color.label2rgb(nucleoli_labels))
axes[0].set_title('Original nucleoli labels')

axes[1].imshow(skimage.color.label2rgb(nuclei_labels))
axes[1].set_title('Nuclei labels')

# select nuclei using nucleoli as a mask
parent_label = np.zeros_like(nucleoli_labels)
parent_label[nucleoli_labels != 0] = nuclei_labels[nucleoli_labels != 0]
axes[2].imshow(skimage.color.label2rgb(parent_label))
axes[2].set_title('Nucleoli labeled by Nuclei')

child_inds = nucleoli_labels[nucleoli_labels != 0]
parent_inds = nuclei_labels[nucleoli_labels != 0]

parent_label = np.zeros_like(nucleoli_labels)
# for each child index, find the most common parent
for ind in np.unique(child_inds):
    vals, counts = np.unique(parent_inds[child_inds == ind], return_counts=True)
    mode_ind = vals[np.argmax(counts)]
    parent_label[nucleoli_labels == ind] = mode_ind
    
axes[3].imshow(skimage.color.label2rgb(parent_label))
axes[3].set_title('Nucleoli labeled by Mode Nuclei')

Notice that nucleoli that are not found in a nucleus are lost in this process.  For child objects that are partly outside a parent will be partly retained in the first case but in the second case, a child object will be lost if it is mostly outside of the parent.

From there, you can add a column to your measurement table of the parent nucleus.

Finally, we will move onto locating the FC locations.  This is challenging because they are fairly close together and difficult to resolve.  We will focus on a single nucleolus but the same procedure can be applied to the entire image.

In [ ]:
cell_image = bioio.BioImage('images/sample.nd2')
# read entire scene to memory.  squeeze removes singleton dimensions (T and Z here)
xcell_image = cell_image.xarray_data.squeeze()

# now we want to determine the min and max values to normalize for display
# passing in X and Y tells xarray we want the min/max over those dimensions.  The result is a min/max for each channel
img_min, img_max = xcell_image.min(['X', 'Y']), xcell_image.max(['X', 'Y'])
# Normalize the image between 0 and 1, xarray takes care of keeping the sizes and columns straight
xcell_normed = (xcell_image - img_min) / (img_max - img_min)

fc_image = xcell_normed.isel(C=2).to_numpy()[750:1000, 600:750]

skimage.filters.try_all_threshold(fc_image, figsize=(10, 8), verbose=False)

We will perform this in two different ways.  The first will use a convolution to enhance objects that are around the expected size for an FC which will allow us to segment the image and measure factors like the area and intensity.  The second will use blob finding, which can be easier and provide good estimates of the count and location of the FCs but is not very good at estimating the size and shape.

In [ ]:
# enhance FC image prior to thresholding
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

axes[0].imshow(fc_image, cmap='gray')
axes[0].set_title('Original Image')

enhanced = skimage.morphology.white_tophat(fc_image, footprint=skimage.morphology.disk(5))
axes[1].imshow(enhanced, cmap='gray')
axes[1].set_title('Ehanced Image')

mask = enhanced > skimage.filters.threshold_otsu(enhanced)
axes[2].set_title('Otsu Threshold')
axes[2].imshow(skimage.color.label2rgb(mask))

mask = skimage.measure.label(mask)
axes[3].set_title('Label objects')
axes[3].imshow(skimage.color.label2rgb(mask))

Not terrible, you may have to play with the size of the disk used in the `white_tophat`, blur or median filter the enhanced image, try different thresholding and filter by size and circularity to get the final segmentatio pattern.  The `white_tophat` has effectively highlighted regions that are around the expected size of an FC.  If you are interested in examining FCs of varied sizes you may have to repeat the procedure at various radii or use a different method.

The `white_tophat` works by subtracting image features that are larger than what you specify.  It performs a morphological opening (erosion follwed by dilation) similar to how we tried to separate nuclei above.  The difference is with a non-binary image, the erosion works by taking the minimum value in the footprint while dilation takes the maximum.  The overall effect of the opening is to remove bright objects smaller than the footprint.  You then subtract the opened image from the original to highlight the small, bright objects.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(fc_image, cmap='gray')
axes[0].set_title('Original Image')

tophat = skimage.morphology.erosion(fc_image, footprint=skimage.morphology.disk(5))
axes[1].imshow(tophat, cmap='gray')
axes[1].set_title('Eroded Image')

tophat = skimage.morphology.dilation(tophat, footprint=skimage.morphology.disk(5))
axes[2].imshow(tophat, cmap='gray')
axes[2].set_title('Opened Image')

axes[3].imshow(fc_image - tophat, cmap='gray')
axes[3].set_title('Tophat Image')

Next, we can use blob finding to try to find the center and approximate radius of each FC.  Starting with the code from skimage documentation:

In [ ]:
blobs_log = skimage.feature.blob_log(fc_image, max_sigma=8, num_sigma=20, threshold=0.05)

# Compute radii in the 3rd column.
blobs_log[:, 2] = blobs_log[:, 2] * np.sqrt(2)

blobs_dog = skimage.feature.blob_dog(fc_image, max_sigma=8, threshold=0.05)
blobs_dog[:, 2] = blobs_dog[:, 2] * np.sqrt(2)

blobs_doh = skimage.feature.blob_doh(fc_image, max_sigma=8, threshold=0.05)

blobs_list = [blobs_log, blobs_dog, blobs_doh]
colors = ['yellow', 'lime', 'red']
titles = ['Laplacian of Gaussian', 'Difference of Gaussian', 'Determinant of Hessian']
sequence = zip(blobs_list, colors, titles)

fig, axes = plt.subplots(1, 3, figsize=(12, 7), sharex=True, sharey=True)
ax = axes.ravel()

for idx, (blobs, color, title) in enumerate(sequence):
    ax[idx].set_title(title)
    ax[idx].imshow(fc_image, cmap='gray')
    for blob in blobs:
        y, x, r = blob
        c = plt.Circle((x, y), r, color=color, linewidth=2, fill=False)
        ax[idx].add_patch(c)
    ax[idx].set_axis_off()

plt.tight_layout()

As is usually the case, the Laplacian of Gaussian has performed the best.  The downside of the `blob_log` detector is it is the slowest of the 3 methods, especially for larger blobs.  Also note that the result is highly dependent on the max sigma (radius of blobs in pixels) and threshold.  Also note that the estimated radius is fairly inaccurate and frequently over estimates the size.  Using a higher `num_sigma` can help, but only slightly.  

The LoG detector works by convolving the input image with a kernel which maximizes response when it matches the underlying image feature.  The particular kernel is a Gaussian filter (which blurs) followed by a Laplacian filter (which highlights edges).  When the sigma of the filter is the same as the blob and at the correct location, the result will be a high value.  To find blobs of any size, the proceedure is repeated with multiple values of sigma and the maximum response is recorded.  This provides an estimate of both the size and location of blobs within the range of sigma values.  Increasing the maximum sigma may cause multiple blobs to group together and increasing the number of sigmas to try provides better estimates of the matching radius.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 5))
axes[0].imshow(fc_image, cmap='gray')
axes[0].set_title('Original Image')

img = skimage.filters.gaussian(fc_image, 4)
axes[1].imshow(img, cmap='gray')
axes[1].set_title('Gaussian sigma = 4')

img = skimage.filters.laplace(img, 4)
axes[2].imshow(img, cmap='gray')
axes[2].set_title('Laplacian of Gaussian\n(sigma=4, ksize=4)')

img = np.zeros((30, 30))
img[15, 15] = 1
axes[3].imshow(skimage.filters.laplace(skimage.filters.gaussian(img, 4), 4), cmap='gray')
axes[3].set_title('Approximate Kernel')